# Online DPD loop — ADRV9026

One saved condition (LO, TX attenuation, ORx gain) from a capture CSV is held
for every iteration; the ORx AGC does not run. Each iteration transmits `u`,
captures, aligns the capture `z` to `u`, and scores it against the original
input `x`. Then `step(x, u, z, it)` returns the next `u`.

`step` is the only model code, and it lives here, not in `adrvtrx`. The
example is an indirect-learning GMP from `dpd_kit`, scaled the same way as the
offline study: peak-normalize `z` and rotate it onto `x`, fit a post-inverse
from `z` to `u` on the samples around the peak, then predistort `x` with its
peak `PEAK_MARGIN_DB` below the peak the post-inverse was trained on.

## 1. Parameters

In [ ]:
from adrvtrx import TxChannel, RxChannel

PROFILE = "ADRV9025Init_StdUseCase98_LinkSharing.profile"

TX_CHANNEL  = TxChannel.TX1
ORX_CHANNEL = RxChannel.ORX1

CONDITIONS_CSV = "captures/operating_sweep/TX1_conditions.csv"
CONDITION_NAME = "TX1_0dB_2200MHz_100BW"

N_ITER = 4
DPD_KIT_SRC = r"C:\Users\ohammi\OneDrive - aus.edu\Scalability_Final_Rev\Polynomials\dpd_kit\src"
K, MEMORY, CROSS = 5, 5, 2
N_TRAIN = 8192
PEAK_MARGIN_DB = 0.2     # input peak sits this far below the peak the post-inverse saw

SAVE_DIR = "DPD_DUT/ila_gmp"
LABEL = "ila_gmp"
OUTPUT_OVERSAMPLE = 2

## 2. Imports, config, profile, condition

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from adrvtrx.config import load_config
from adrvtrx.profile import read_profile
from adrvtrx.radio import Radio
from adrvtrx.experiment import verify_status
from adrvtrx.waveform import full_scale

import sys

from adrvtrx.conditions import load_aligned_iq, load_conditions
from adrvtrx.linearize import linearize
from adrvtrx.metrics import nmse_db

if DPD_KIT_SRC not in sys.path:
    sys.path.insert(0, DPD_KIT_SRC)
from dpd_kit.models import GMP
from dpd_kit.sampling import window
from dpd_kit.signals.core import Signal

cfg = load_config()
cfg.profile_name = PROFILE
info = read_profile(cfg.profile_path)
FS = info.orx_rate_hz

csv_dir = Path(CONDITIONS_CSV).parent
condition = next(r for r in load_conditions(CONDITIONS_CSV) if r.name == CONDITION_NAME)
x = load_aligned_iq(csv_dir / condition.in_file)
cfg.lo.lo1_hz = int(condition.freq_hz)
print(f"{condition.name}: atten {condition.tx_atten_db:.2f} dB  gain {condition.orx_gain}  "
      f"NMSE without DPD {condition.nmse_db:.2f} dB")

## 3. The step

All three inputs are normalized floats of the same length; `z` is aligned to
`u`. Return the next waveform, or `None` to stop early. Samples above 1.0 are
clipped at the DAC and reported as `tx_clipped`.

In [ ]:
def ila_gmp_step(x, u, z, it):
    phase = np.exp(1j * np.angle(np.vdot(x, z)))
    zs = Signal(z / np.max(np.abs(z)) / phase, fs=FS, fc=float(condition.freq_hz), name="z")
    us = Signal(u, fs=FS, fc=float(condition.freq_hz), name="u")
    model = GMP.unified(K=K, N=MEMORY, M=CROSS)
    train_idx, _ = window(
        zs, us, scheme="contiguous", n_train=N_TRAIN,
        memory=model.memory, stratify_on="input",
    )
    lo = int(train_idx[0]) - (model.memory - 1)
    hi = int(train_idx[-1]) + 1
    model.fit(
        Signal(zs.iq[lo:hi], fs=FS, fc=zs.fc, name="z_train"),
        Signal(us.iq[lo:hi], fs=FS, fc=us.fc, name="u_train"),
    )
    x_in = x / np.max(np.abs(x)) * 10 ** (-PEAK_MARGIN_DB / 20)
    u_next = model.predict(Signal(x_in, fs=FS, fc=zs.fc, name="x")).numpy
    print(f"    it {it}: post-inverse NMSE "
          f"{nmse_db(u[MEMORY:], model.predict(zs).numpy[MEMORY:]):.2f} dB, "
          f"next peak {np.max(np.abs(u_next)):.3f}")
    return u_next

## 4. Connect and program

In [ ]:
radio = Radio(cfg)
radio.connect()
radio.force_safe()
radio.program()

radio.set_tx_atten(TX_CHANNEL, condition.tx_atten_db)
lock = radio.retune_lo("LO1", int(condition.freq_hz))

for k, v in verify_status(radio).items():
    print(f"{k}: {v}")
print(f"LO1 = {radio.get_lo('LO1')/1e6:.3f} MHz   lock 0x{lock:X}")
print(f"TX1 atten = {radio.get_tx_atten(TX_CHANNEL):.2f} dB")

## 5. Run the loop

In [ ]:
def show(it, rec):
    print(f"it {it}: NMSE {rec.nmse_db:.2f} dB  PAPR comp {rec.papr_compression_db:.2f} dB  "
          f"ACLR {rec.aclr_lower_dbc:.2f}/{rec.aclr_upper_dbc:.2f} dBc  "
          f"railed {rec.railed}  tx clipped {rec.tx_clipped}")

result = linearize(
    radio, condition, x, ila_gmp_step,
    tx=TX_CHANNEL, orx=ORX_CHANNEL, tx_bits=info.tx_bits, rx_bits=info.rx_bits, fs=FS,
    n_iter=N_ITER, save_dir=SAVE_DIR, label=LABEL, oversample=OUTPUT_OVERSAMPLE,
    on_iter=show,
)
print(f"stopped: {result.reason}")

## Safe-state and disconnect

In [ ]:
radio.safe_state()
radio.disconnect()
print("TX safe, board disconnected")

## 7. Iterations and spectrum

In [ ]:
def psd_db(s):
    n = len(s)
    p = np.abs(np.fft.fftshift(np.fft.fft(s * np.hanning(n)))) ** 2
    f = np.fft.fftshift(np.fft.fftfreq(n, 1.0 / FS)) / 1e6
    return f, 10 * np.log10(p / p.max())

recs = result.records
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
it = np.arange(len(recs))
ax[0].plot(it, [r.nmse_db for r in recs], "o-", label="NMSE (dB)")
ax[0].plot(it, [max(r.aclr_lower_dbc, r.aclr_upper_dbc) for r in recs], "o-",
           label="worst ACLR (dBc)")
ax[0].set_xlabel("iteration"); ax[0].legend(); ax[0].grid(True)

z0 = load_aligned_iq(Path(SAVE_DIR) / recs[0].out_file)
for s, label in ((x, "input"), (z0, "without DPD"), (result.z, "last iteration")):
    f, p = psd_db(s)
    ax[1].plot(f, p, lw=0.6, label=label)
ax[1].set_xlim(-FS / 2e6, FS / 2e6); ax[1].set_ylim(-80, 3)
ax[1].set_xlabel("MHz"); ax[1].set_ylabel("dB rel. peak"); ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()